In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/adaptive-lora/checkpoints/adaptive', exist_ok=True)
print("Drive ready")

Mounted at /content/drive
Drive ready


In [ ]:
%%capture
!pip install transformers peft accelerate bitsandbytes datasets trl

In [ ]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset
from torch.utils.data import DataLoader

print("Imports done")

Imports done


In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Loaded. Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
with open('/content/drive/MyDrive/adaptive-lora/results/rank_allocation.json', 'r') as f:
    rank_allocation = json.load(f)

print(f"Loaded {len(rank_allocation)} rank assignments")
print("\nSample ranks:")
sorted_ranks = sorted(rank_allocation.items(), key=lambda x: x[1], reverse=True)
for name, rank in sorted_ranks[:5]:
    print(f"  {name}: rank {rank}")
print("  ...")
for name, rank in sorted_ranks[-5:]:
    print(f"  {name}: rank {rank}")

Loaded 64 rank assignments

Sample ranks:
  layer_0_v_proj: rank 16
  layer_1_v_proj: rank 16
  layer_2_v_proj: rank 16
  layer_3_v_proj: rank 16
  layer_4_v_proj: rank 16
  ...
  layer_29_q_proj: rank 4
  layer_30_q_proj: rank 4
  layer_30_v_proj: rank 4
  layer_0_q_proj: rank 2
  layer_1_q_proj: rank 2


In [ ]:
model = prepare_model_for_kbit_training(model)

# convert your rank_allocation keys to match PEFT's expected format
# your keys look like: "layer_0_v_proj"
# PEFT's rank_pattern expects: "model.layers.0.self_attn.v_proj"

rank_pattern = {}
for key, rank in rank_allocation.items():
    # parse "layer_0_v_proj" → layer number and projection type
    parts = key.split('_')
    layer_num = parts[1]
    proj_type = parts[2] + '_' + parts[3]  # "v_proj" or "q_proj"

    # build the PEFT-expected key format
    peft_key = f"model.layers.{layer_num}.self_attn.{proj_type}"
    rank_pattern[peft_key] = rank

print(f"rank_pattern built: {len(rank_pattern)} entries")
print("\nSample rank_pattern entries:")
sample = list(rank_pattern.items())[:3]
for k, v in sample:
    print(f"  {k}: {v}")

# apply LoRA with per-layer ranks
lora_config = LoraConfig(
    r=8,                          # default fallback rank
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    rank_pattern=rank_pattern,    # ← THIS is the key difference
    alpha_pattern={               # alpha = 2x rank per layer
        k: v * 2 for k, v in rank_pattern.items()
    }
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

rank_pattern built: 64 entries

Sample rank_pattern entries:
  model.layers.0.self_attn.q_proj: 2
  model.layers.0.self_attn.v_proj: 16
  model.layers.1.self_attn.q_proj: 2
trainable params: 3,014,656 || all params: 7,244,746,752 || trainable%: 0.0416


In [ ]:
dataset = load_dataset("fancyzhx/ag_news")

def format_example(example):
    label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
    label = label_names[example['label']]
    text = f"""Classify the following news article into one of these categories: World, Sports, Business, Sci/Tech.

Article: {example['text']}

Category: {label}"""
    return {"text": text}

formatted_train = dataset['train'].map(format_example)
formatted_test = dataset['test'].map(format_example)

TRAIN_SIZE = 8000
train_subset = formatted_train.shuffle(seed=42).select(range(TRAIN_SIZE))

print(f"Train: {len(train_subset)}")

In [ ]:
MAX_LENGTH = 256

def tokenize(example):
    result = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_train = train_subset.map(tokenize, remove_columns=train_subset.column_names)
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
print(f"Tokenized {len(tokenized_train)} examples")

In [ ]:
train_loader = DataLoader(tokenized_train, batch_size=4, shuffle=True)

EPOCHS = 2
total_steps = len(train_loader) * EPOCHS
print(f"Batches per epoch: {len(train_loader)}")
print(f"Total steps: {total_steps}")

Batches per epoch: 2000
Total steps: 4000


In [ ]:
import os

CHECKPOINT_EVERY = 500
SAVE_PATH = '/content/drive/MyDrive/adaptive-lora/checkpoints/adaptive'
os.makedirs(SAVE_PATH, exist_ok=True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4
)

checkpoint_file = os.path.join(SAVE_PATH, 'training_state.pt')

if os.path.exists(checkpoint_file):
    print("Checkpoint found — resuming...")
    checkpoint = torch.load(checkpoint_file)
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_step = checkpoint['step']
    losses = checkpoint['losses']
    print(f"Resuming from step {start_step}/{total_steps}")
else:
    print("Starting fresh")
    start_step = 0
    losses = []

model.train()
step = 0

for epoch in range(EPOCHS):
    print(f"\n=== EPOCH {epoch+1}/{EPOCHS} ===")
    for batch in tqdm(train_loader):
        step += 1
        if step <= start_step:
            continue

        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        losses.append(loss.item())

        if step % 100 == 0:
            avg_loss = np.mean(losses[-100:])
            print(f"Step {step}/{total_steps} | Loss: {avg_loss:.4f}")

        if step % CHECKPOINT_EVERY == 0:
            model.save_pretrained(SAVE_PATH)
            torch.save({
                'step': step,
                'optimizer_state': optimizer.state_dict(),
                'losses': losses,
            }, checkpoint_file)
            print(f"  Checkpoint saved at step {step}")

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
torch.save({
    'step': step,
    'optimizer_state': optimizer.state_dict(),
    'losses': losses,
}, checkpoint_file)

with open('/content/drive/MyDrive/adaptive-lora/results/adaptive_losses.json', 'w') as f:
    json.dump(losses, f)

print(f"\nAdaptive training complete")

Starting fresh

=== EPOCH 1/2 ===


  0%|          | 0/2000 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
  5%|▌         | 100/2000 [06:52<2:12:10,  4.17s/it]

Step 100/4000 | Loss: 0.6274


 10%|█         | 200/2000 [13:47<2:04:37,  4.15s/it]

Step 200/4000 | Loss: 0.5042


...

100%|█████████▉| 1999/2000 [2:19:46<00:04,  4.22s/it]

Step 4000/4000 | Loss: 0.4392


100%|██████████| 2000/2000 [2:19:51<00:00,  4.20s/it]

  Checkpoint saved at step 4000



Adaptive training complete
